# scTASL tutorial: cell-type-specific integration fine-tuning

This notebook fine-tunes a pretrained paired RNA-ATAC integration model on one
cell type. It starts from the full integration outputs saved by
`1_integration.ipynb` and writes GRN-ready fine-tuned embeddings under
`results/<DATASET>/GRN/<cell_type>/`.

**Required inputs:**

- `results/<DATASET>/integration/adata/rna_emb.h5ad`
- `results/<DATASET>/integration/adata/atac_emb.h5ad`
- `results/<DATASET>/integration/Integration_final_model.pt`
- `dataset/<DATASET>/graph_data.pkl`, the prior graph used by the full integration run

Both omics inputs must contain a `counts` layer. This fine-tuning example also
uses `obs["cell_type"]` to select one annotated cell type; unlabeled datasets
need a different subset definition. Paired cells must have identical observation
names in identical order.

> Run this notebook from the repository root. Only load pickle and checkpoint
> files that were produced by this project or another trusted source.


## 1. Setup

Import only the packages required by this workflow. Keeping imports explicit
makes the tutorial easier to audit and avoids hidden namespace dependencies.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from utils.common.data_configure import load_omics_inputs
from utils.integration.finetune import finetune_single_cell_type


### Configure the fine-tuning run

Select one value shown by the cell-type summary below. Stage 1 updates the
modality encoders and decoders while optionally freezing the graph branch.
Stage 2 unfreezes the graph branch so feature embeddings can adapt.


In [ ]:
DATASET = "PBMC-10k"
TASK = "GRN"
TARGET_CELL_TYPE = "CD14 Mono"
CELL_TYPE_KEY = "cell_type"
IS_PAIRED = True
RANDOM_SEED = 0

LEARNING_RATE = 2e-4
STAGE1_EPOCHS = 50
STAGE2_EPOCHS = 20
FREEZE_GRAPH_STAGE1 = True
PRINT_EVERY = 10

DATA_DIR = Path("dataset") / DATASET
INTEGRATION_DIR = Path("results") / DATASET / "integration"
INTEGRATION_ADATA_DIR = INTEGRATION_DIR / "adata"
SAFE_CELL_TYPE = TARGET_CELL_TYPE.replace(" ", "_").replace("/", "_")
OUTPUT_DIR = Path("results") / DATASET / TASK / SAFE_CELL_TYPE
BASE_CHECKPOINT = INTEGRATION_DIR / "Integration_final_model.pt"

INPUT_PATHS = {
    "RNA": INTEGRATION_ADATA_DIR / "rna_emb.h5ad",
    "ATAC": INTEGRATION_ADATA_DIR / "atac_emb.h5ad",
    "graph": DATA_DIR / "graph_data.pkl",
    "checkpoint": BASE_CHECKPOINT,
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print(f"Target cell type: {TARGET_CELL_TYPE}")
print(f"Integration input directory: {INTEGRATION_DIR.resolve()}")
print(f"Base checkpoint: {BASE_CHECKPOINT}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Compute device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")


## 2. Load and validate the inputs

These checks stop the run before training when files, annotations, count
layers, or paired-cell ordering are inconsistent. The table also helps the
user choose a cell type with enough cells in both omics inputs.


In [ ]:
# Fine-tuning loads the full integration embeddings and checkpoint, then reuses the matching prior graph.
# This example requires cell_type because TARGET_CELL_TYPE selects an annotated group.
rna, atac, graph_data, input_summary = load_omics_inputs(
    rna_path=INPUT_PATHS["RNA"],
    atac_path=INPUT_PATHS["ATAC"],
    graph_path=INPUT_PATHS["graph"],
    additional_required_paths={"checkpoint": INPUT_PATHS["checkpoint"]},
    cell_type_key=CELL_TYPE_KEY,
    require_counts=True,
    require_cell_type=True,
    require_paired=IS_PAIRED,
    paired_task="paired fine-tuning",
)

display(input_summary)

cell_type_counts = pd.concat(
    {
        "RNA cells": rna.obs[CELL_TYPE_KEY].value_counts(),
        "ATAC cells": atac.obs[CELL_TYPE_KEY].value_counts(),
    },
    axis=1,
).fillna(0).astype(int).sort_index()
display(cell_type_counts)

if TARGET_CELL_TYPE not in cell_type_counts.index:
    raise ValueError(
        f'TARGET_CELL_TYPE="{TARGET_CELL_TYPE}" is not present in the inputs.'
    )
if (cell_type_counts.loc[TARGET_CELL_TYPE] == 0).any():
    raise ValueError(
        f'Cell type "{TARGET_CELL_TYPE}" must be present in both omics.'
    )

rna_target_ids = rna.obs_names[rna.obs[CELL_TYPE_KEY] == TARGET_CELL_TYPE]
atac_target_ids = atac.obs_names[atac.obs[CELL_TYPE_KEY] == TARGET_CELL_TYPE]
if IS_PAIRED and not rna_target_ids.equals(atac_target_ids):
    raise ValueError(
        "The selected RNA and ATAC cells are not paired in identical order."
    )


## 3. Fine-tune the selected cell type

The helper subsets both omics inputs, restores the pretrained full-cell model,
performs the two training stages, attaches cell and feature embeddings, and
saves the fine-tuned AnnData objects when `save_h5ad=True`.


In [ ]:
fine_tune_result = finetune_single_cell_type(
    rna=rna,
    atac=atac,
    graph_data=graph_data,
    base_ckpt_path=str(BASE_CHECKPOINT),
    output_dir=str(OUTPUT_DIR),
    target_cell_type=TARGET_CELL_TYPE,
    cell_type_key=CELL_TYPE_KEY,
    use_obs_names=True,
    is_paired=IS_PAIRED,
    lr=LEARNING_RATE,
    stage1_epochs=STAGE1_EPOCHS,
    stage2_epochs=STAGE2_EPOCHS,
    freeze_graph_stage1=FREEZE_GRAPH_STAGE1,
    print_every=PRINT_EVERY,
    save_h5ad=True,
)


## 4. Check and save the fine-tuned outputs

A prototype is the mean cell embedding for the selected cell type. The two
modality prototypes are saved together for later GRN or similarity analyses.


In [ ]:
fine_tuned_adatas = fine_tune_result["adatas"]
prototypes = fine_tune_result["prototypes"]

output_summary = pd.DataFrame(
    {
        "cells": fine_tune_result["n_cells"],
        "cell_embedding_shape": {
            name: adata_obj.obsm["embedding"].shape
            for name, adata_obj in fine_tuned_adatas.items()
        },
        "feature_embedding_shape": {
            name: adata_obj.varm["feature_embedding"].shape
            for name, adata_obj in fine_tuned_adatas.items()
        },
        "prototype_shape": {
            name: prototype.shape for name, prototype in prototypes.items()
        },
    }
)
display(output_summary)

prototype_path = OUTPUT_DIR / f"prototypes_{SAFE_CELL_TYPE}.npz"
np.savez_compressed(
    prototype_path,
    RNA=prototypes["RNA"],
    ATAC=prototypes["ATAC"],
)

print("Saved AnnData files:")
for omics_name, path in fine_tune_result["saved_h5ad_paths"].items():
    print(f"- {omics_name}: {path}")
print(f"Saved prototypes: {prototype_path}")


## 5. Next steps

Use the fine-tuned feature embeddings and prototypes in the downstream GRN
notebooks. To fine-tune another cell type, change `TARGET_CELL_TYPE`, review
its RNA/ATAC counts, and rerun the notebook from the top. Fine-tuning very
small cell populations can produce unstable embeddings, so compare cell
counts and training logs before interpreting downstream networks.
